In [1]:
import os
import json
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import torchvision.models as models
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc
)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

In [2]:
DRIVE_DATA_DIR = "/content/drive/MyDrive/rvf10k"

In [3]:
BATCH_SIZE = 32
LEARNING_RATE = 0.0003
EPOCHS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

cuda


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
train_path = os.path.join(DRIVE_DATA_DIR, 'train')
val_path = os.path.join(DRIVE_DATA_DIR, 'valid')

if not os.path.exists(train_path) or not os.path.exists(val_path):
  raise FileNotFoundError(f"Could not find 'train'/'valid' inside '{DRIVE_DATA_DIR}'. Check your path.")

In [6]:
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

In [7]:
class CustomCNN(nn.Module):
    """A small CNN built from scratch (Week 4 material) — our baseline."""
    def __init__(self):
        super(CustomCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 16 x 112 x 112

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 32 x 56 x 56

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)   # 64 x 28 x 28
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 28 * 28, 128),
            nn.ReLU(),
            nn.Dropout(p=0.4),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [8]:
 def build_transfer_learning_model():
    """Pretrained ResNet18 with a frozen backbone and a new 2-class head."""
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for param in model.parameters():
        param.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model

def count_trainable_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

In [9]:
def run_training_loop(model, optimizer, criterion, model_name, train_loader, val_loader):
    print(f"\n--- Training: {model_name} ---")
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    start_time = time.time()
    for epoch in range(EPOCHS):
        epoch_start = time.time()
        #Training the model
        model.train()
        running_loss, correct_train, total_train = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct_train += torch.sum(preds == labels.data)
            total_train += images.size(0)

        epoch_train_loss = running_loss / total_train
        epoch_train_acc = (correct_train.double() / total_train).item()
        #validation of model
        model.eval()
        running_val_loss, correct_val, total_val = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                loss = criterion(outputs, labels)

                running_val_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                correct_val += torch.sum(preds == labels.data)
                total_val += images.size(0)
        epoch_val_loss = running_val_loss / total_val
        epoch_val_acc = (correct_val.double() / total_val).item()
        duration = time.time() - epoch_start
        print(f"Epoch {epoch+1}/{EPOCHS} [{duration:.1f}s] -> "
              f"Loss: {epoch_train_loss:.4f} | Acc: {epoch_train_acc:.4f} || "
              f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)

    total_train_time = time.time() - start_time
    print(f"--- Finished {model_name} in {total_train_time:.1f}s ---")
    return history, total_train_time

In [10]:
def evaluate_model(model, val_loader, val_dataset, classes, model_name, history, train_time):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    # Time a single inference pass over the val set (for speed comparison)
    inf_start = time.time()
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)[:, 1]  # prob of "fake" class
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
    inf_time = time.time() - inf_start
    avg_inf_time_per_image = inf_time / len(val_dataset)

    report_dict = classification_report(all_labels, all_preds, target_names=classes, output_dict=True)
    print(f"\n=== {model_name} Classification Report ===")
    print(classification_report(all_labels, all_preds, target_names=classes))

    # --- Learning curves ---
    epochs_range = range(1, EPOCHS + 1)
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, history['train_loss'], label='Train Loss', marker='o')
    plt.plot(epochs_range, history['val_loss'], label='Val Loss', marker='x')
    plt.title(f'{model_name} - Loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, history['train_acc'], label='Train Acc', marker='o')
    plt.plot(epochs_range, history['val_acc'], label='Val Acc', marker='x')
    plt.title(f'{model_name} - Accuracy')
    plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{model_name.lower().replace(' ', '_')}_curves.png"))
    plt.close()

    # --- Confusion matrix ---
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='mako', xticklabels=classes, yticklabels=classes)
    plt.title(f'{model_name} Confusion Matrix')
    plt.ylabel('Actual'); plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{model_name.lower().replace(' ', '_')}_confusion_matrix.png"))
    plt.close()

    # --- ROC curve ---
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(5, 5))
    plt.plot(fpr, tpr, label=f'{model_name} (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title(f'{model_name} ROC Curve')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{model_name.lower().replace(' ', '_')}_roc.png"))
    plt.close()

    # --- Misclassified samples grid (for error analysis section) ---
    save_misclassified_grid(model, val_loader, val_dataset, classes, model_name)

    total_params, trainable_params = count_trainable_params(model)

    return {
        "model_name": model_name,
        "val_accuracy": report_dict['accuracy'],
        "macro_f1": report_dict['macro avg']['f1-score'],
        "macro_precision": report_dict['macro avg']['precision'],
        "macro_recall": report_dict['macro avg']['recall'],
        "roc_auc": roc_auc,
        "total_params": total_params,
        "trainable_params": trainable_params,
        "total_training_time_sec": round(train_time, 2),
        "avg_inference_time_per_image_ms": round(avg_inf_time_per_image * 1000, 3),
        "history": history,
        "fpr": fpr.tolist(),
        "tpr": tpr.tolist(),
    }

In [11]:
def save_misclassified_grid(model, val_loader, val_dataset, classes, model_name, n_samples=8):
    """Save a grid of misclassified images — great for the error-analysis
    section of the report and easy to talk about in the video."""
    model.eval()
    wrong_images, wrong_true, wrong_pred = [], [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images_dev = images.to(DEVICE)
            outputs = model(images_dev)
            _, preds = torch.max(outputs, 1)
            preds_cpu = preds.cpu()
            mismatch = preds_cpu != labels
            for i in range(len(labels)):
                if mismatch[i] and len(wrong_images) < n_samples:
                    wrong_images.append(images[i])
                    wrong_true.append(labels[i].item())
                    wrong_pred.append(preds_cpu[i].item())
            if len(wrong_images) >= n_samples:
                break

    if not wrong_images:
        print(f"No misclassified samples found for {model_name} (unexpectedly perfect val set).")
        return

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    cols = 4
    rows = int(np.ceil(len(wrong_images) / cols))
    plt.figure(figsize=(cols * 3, rows * 3))
    for idx, img_tensor in enumerate(wrong_images):
        img = img_tensor.permute(1, 2, 0).numpy()
        img = std * img + mean
        img = np.clip(img, 0, 1)
        plt.subplot(rows, cols, idx + 1)
        plt.imshow(img)
        plt.title(f"True: {classes[wrong_true[idx]]}\nPred: {classes[wrong_pred[idx]]}", fontsize=9)
        plt.axis('off')
    plt.suptitle(f"{model_name} — Misclassified Examples")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{model_name.lower().replace(' ', '_')}_misclassified.png"))
    plt.close()

In [12]:
def plot_combined_comparison(results_list):
    """Overlay both models' curves on shared axes — the key comparison
    figure for your report."""
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    for r in results_list:
        plt.plot(range(1, EPOCHS + 1), r['history']['val_loss'], marker='o', label=r['model_name'])
    plt.title('Validation Loss Comparison')
    plt.xlabel('Epoch'); plt.ylabel('Val Loss'); plt.legend()

    plt.subplot(1, 2, 2)
    for r in results_list:
        plt.plot(range(1, EPOCHS + 1), r['history']['val_acc'], marker='o', label=r['model_name'])
    plt.title('Validation Accuracy Comparison')
    plt.xlabel('Epoch'); plt.ylabel('Val Accuracy'); plt.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "combined_validation_comparison.png"))
    plt.close()

    # Combined ROC comparison
    plt.figure(figsize=(6, 6))
    for r in results_list:
        plt.plot(r['fpr'], r['tpr'], label=f"{r['model_name']} (AUC={r['roc_auc']:.3f})")
    plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title('ROC Curve Comparison')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "combined_roc_comparison.png"))
    plt.close()


In [21]:
OUTPUT_DIR = "/content/drive/MyDrive/results"

In [22]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])

In [23]:
train_dataset = ImageFolder(root=train_path, transform=train_transform)
val_dataset = ImageFolder(root=val_path, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [24]:
classes = train_dataset.classes
print(f"Classes: {classes}")
print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

loss_function = nn.CrossEntropyLoss()
all_results = []

Classes: ['fake', 'real']
Train size: 7000 | Val size: 3000


In [25]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

#model1 = cnn building from scratch
scratch_model = CustomCNN().to(DEVICE)
scratch_optimizer = optim.Adam(scratch_model.parameters(), lr=LEARNING_RATE)
scratch_history, scratch_time = run_training_loop(scratch_model, scratch_optimizer, loss_function, "Custom CNN (Scratch)", train_loader, val_loader)
scratch_results = evaluate_model(scratch_model, val_loader, val_dataset, classes, "Custom CNN (Scratch)", scratch_history, scratch_time)
torch.save(scratch_model.state_dict(), os.path.join(OUTPUT_DIR, "custom_cnn_scratch.pth"))
all_results.append(scratch_results)


--- Training: Custom CNN (Scratch) ---
Epoch 1/5 [96.1s] -> Loss: 0.7374 | Acc: 0.6021 || Val Loss: 0.6217 | Val Acc: 0.6657
Epoch 2/5 [67.7s] -> Loss: 0.6338 | Acc: 0.6167 || Val Loss: 0.5853 | Val Acc: 0.7087
Epoch 3/5 [65.1s] -> Loss: 0.6164 | Acc: 0.6190 || Val Loss: 0.5714 | Val Acc: 0.7240
Epoch 4/5 [65.1s] -> Loss: 0.5935 | Acc: 0.6354 || Val Loss: 0.5386 | Val Acc: 0.7420
Epoch 5/5 [65.0s] -> Loss: 0.5709 | Acc: 0.6367 || Val Loss: 0.5268 | Val Acc: 0.7397
--- Finished Custom CNN (Scratch) in 359.0s ---

=== Custom CNN (Scratch) Classification Report ===
              precision    recall  f1-score   support

        fake       0.75      0.71      0.73      1500
        real       0.73      0.77      0.75      1500

    accuracy                           0.74      3000
   macro avg       0.74      0.74      0.74      3000
weighted avg       0.74      0.74      0.74      3000



In [26]:
#model2: using tranfer learning (ResNet18)
transfer_model = build_transfer_learning_model().to(DEVICE)
#adam optimiser
transfer_optimizer = optim.Adam(filter(lambda p: p.requires_grad, transfer_model.parameters()), lr=LEARNING_RATE)
transfer_history, transfer_time = run_training_loop(transfer_model, transfer_optimizer, loss_function, "Transfer Learning ResNet18", train_loader, val_loader)
transfer_results = evaluate_model(transfer_model, val_loader, val_dataset, classes, "Transfer Learning ResNet18", transfer_history, transfer_time)
torch.save(transfer_model.state_dict(), os.path.join(OUTPUT_DIR, "transfer_resnet18.pth"))
all_results.append(transfer_results)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 175MB/s]



--- Training: Transfer Learning ResNet18 ---
Epoch 1/5 [66.1s] -> Loss: 0.6190 | Acc: 0.6570 || Val Loss: 0.5502 | Val Acc: 0.7327
Epoch 2/5 [68.1s] -> Loss: 0.5348 | Acc: 0.7377 || Val Loss: 0.5467 | Val Acc: 0.7317
Epoch 3/5 [65.5s] -> Loss: 0.5076 | Acc: 0.7549 || Val Loss: 0.5005 | Val Acc: 0.7633
Epoch 4/5 [65.8s] -> Loss: 0.4963 | Acc: 0.7670 || Val Loss: 0.4916 | Val Acc: 0.7720
Epoch 5/5 [65.7s] -> Loss: 0.4830 | Acc: 0.7737 || Val Loss: 0.4844 | Val Acc: 0.7737
--- Finished Transfer Learning ResNet18 in 331.3s ---

=== Transfer Learning ResNet18 Classification Report ===
              precision    recall  f1-score   support

        fake       0.79      0.75      0.77      1500
        real       0.76      0.80      0.78      1500

    accuracy                           0.77      3000
   macro avg       0.77      0.77      0.77      3000
weighted avg       0.77      0.77      0.77      3000



In [27]:
plot_combined_comparison(all_results)

In [28]:
summary = []
for r in all_results:
    summary.append({
        "Model": r["model_name"],
        "Val Accuracy": round(r["val_accuracy"] * 100, 2),
        "Macro F1": round(r["macro_f1"], 4),
        "Macro Precision": round(r["macro_precision"], 4),
        "Macro Recall": round(r["macro_recall"], 4),
        "ROC AUC": round(r["roc_auc"], 4),
        "Total Params": r["total_params"],
        "Trainable Params": r["trainable_params"],
        "Training Time (s)": r["total_training_time_sec"],
        "Inference Time/Image (ms)": r["avg_inference_time_per_image_ms"],
    })

summary_df = pd.DataFrame(summary)
summary_df.to_csv(os.path.join(OUTPUT_DIR, "results_summary.csv"), index=False)


In [29]:
with open(os.path.join(OUTPUT_DIR, "results_summary.json"), "w") as f:
    json.dump(all_results, f, indent=2)

In [30]:
print("\n" + "=" * 70)
print("FINAL COMPARISON TABLE")
print("=" * 70)
print(summary_df.to_string(index=False))
print("\nAll plots, model weights, and results_summary.csv/json saved to:")
print(OUTPUT_DIR)


FINAL COMPARISON TABLE
                     Model  Val Accuracy  Macro F1  Macro Precision  Macro Recall  ROC AUC  Total Params  Trainable Params  Training Time (s)  Inference Time/Image (ms)
      Custom CNN (Scratch)         73.97    0.7395           0.7404        0.7397   0.8196       6446722           6446722             358.96                      5.532
Transfer Learning ResNet18         77.37    0.7736           0.7742        0.7737   0.8507      11177538              1026             331.30                      6.239

All plots, model weights, and results_summary.csv/json saved to:
/content/drive/MyDrive/results
